In [ ]:
pip install faiss-cpu

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# 1️⃣ Documents
docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "AI is used in healthcare and finance."
]

# 2️⃣ Create embeddings
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(docs)
embeddings = np.array(embeddings).astype("float32")
faiss.normalize_L2(embeddings)

# 3️⃣ FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

# 4️⃣ Query
query = "What is machine learning?"
query_vector = model.encode([query])
query_vector = np.array(query_vector).astype("float32")
faiss.normalize_L2(query_vector)

scores, indices = index.search(query_vector, 2)

# 5️⃣ Retrieve context
context = " ".join([docs[i] for i in indices[0]])

# 6️⃣ LLM generator (using pipeline)
generator = pipeline("text-generation", model="gpt2")

prompt = f"Don't add duplicate answer,Give me and Answer the question using the context.\n\nContext: {context}\n\nQuestion: {query}\nAnswer:"

result = generator(prompt, max_length=100)

print(result[0]["generated_text"])